In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("all_month.csv")

print("Dataset shape:", df.shape)

display(df.head())
display(df.tail())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
display(df.describe())

In [ ]:
# Missing-value report
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage":
        (df.isnull().sum() / len(df) * 100).round(2)
})

display(
    missing_report.sort_values(
        "missing_percentage",
        ascending=False
    )
)

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

categorical_columns = [
    "magType",
    "type",
    "status",
    "locationSource",
    "magSource"
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))
    
    
range_columns = [
    "latitude",
    "longitude",
    "depth",
    "mag",
    "nst",
    "gap",
    "dmin",
    "rms"
]

for column in range_columns:
    print(
        column,
        "Min:", df[column].min(),
        "Max:", df[column].max()
    )

In [ ]:
df_clean = df.copy()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.replace(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        regex=True
    )
    .str.lower()
)

print(df_clean.columns.tolist())

text_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

df_clean["time"] = pd.to_datetime(
    df_clean["time"],
    errors="coerce",
    utc=True
)

df_clean["updated"] = pd.to_datetime(
    df_clean["updated"],
    errors="coerce",
    utc=True
)

numeric_columns = [
    "latitude",
    "longitude",
    "depth",
    "mag",
    "nst",
    "gap",
    "dmin",
    "rms",
    "horizontal_error",
    "depth_error",
    "mag_error",
    "mag_nst"
]

for column in numeric_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

print(
    "Duplicates before cleaning:",
    df_clean.duplicated().sum()
)

df_clean = (
    df_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Duplicates after cleaning:",
    df_clean.duplicated().sum()
)

In [ ]:
invalid_latitude = (
    df_clean["latitude"].notna()
    & ~df_clean["latitude"].between(-90, 90)
)

print(
    "Invalid latitude:",
    invalid_latitude.sum()
)
invalid_longitude = (
    df_clean["longitude"].notna()
    & ~df_clean["longitude"].between(-180, 180)
)

print(
    "Invalid longitude:",
    invalid_longitude.sum()
)
invalid_gap = (
    df_clean["gap"].notna()
    & ~df_clean["gap"].between(0, 360)
)

print(
    "Invalid gap values:",
    invalid_gap.sum()
)
error_columns = [
    "horizontal_error",
    "depth_error",
    "mag_error",
    "rms"
]

for column in error_columns:
    negatives = (
        df_clean[column].notna()
        & (df_clean[column] < 0)
    )

    print(
        column,
        "negative values:",
        negatives.sum()
    )
print(
    df_clean["mag"].describe()
)

print(
    df_clean["depth"].describe()
)

print(
    "Negative depth observations:",
    (df_clean["depth"] < 0).sum()
)
Q1 = df_clean["mag"].quantile(0.25)
Q3 = df_clean["mag"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

magnitude_outliers = df_clean[
    (df_clean["mag"] < lower_bound)
    |
    (df_clean["mag"] > upper_bound)
]

print(
    "Potential magnitude outliers:",
    len(magnitude_outliers)
)